In [84]:
import sys
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
sys.path.append('..')
from metric import precision_at_recall
RANDOM_STATE = 42

train = pd.read_csv('../data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('../data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('../data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


### Data overview

In [85]:
train.info()
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 11091 entries, 0 to 11090
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   cookie_id          11091 non-null  str           
 1   cookie_created_at  11091 non-null  datetime64[us]
 2   window_start_ts    11091 non-null  datetime64[us]
 3   window_end_ts      11091 non-null  datetime64[us]
 4   target             11091 non-null  int64         
dtypes: datetime64[us](3), int64(1), str(1)
memory usage: 433.4 KB
<class 'pandas.DataFrame'>
RangeIndex: 328905 entries, 0 to 328904
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   cookie_id      328905 non-null  str           
 1   event_ts       328905 non-null  datetime64[us]
 2   eid            328905 non-null  int64         
 3   event_name     328905 non-null  str           
 4   platform       328905 non-null  st

In [86]:
events['event_name'].value_counts()


event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64

### События внутри заданного окна

In [87]:
columns_to_keep = ['cookie_id', 'window_start_ts','window_end_ts' ]
windows = pd.concat([train[columns_to_keep], test[columns_to_keep]],ignore_index=True)

In [88]:
events_with_windows = pd.merge(windows,events,on='cookie_id',how='right')

In [89]:
mask = (events_with_windows['window_start_ts'] <= events_with_windows['event_ts']) & (events_with_windows['event_ts'] < events_with_windows['window_end_ts'])
events_filtered = events_with_windows[mask]

In [90]:
events.shape, events_filtered.shape

((328905, 14), (288126, 16))

## Признаки

В качестве начальных признаков берем:
- общее число событий
- число уникальных объявлений
- количество событий каждого типа
- число уникальных категорий
- число уникальных локаций

In [91]:
n_unique_items = events_filtered.groupby('cookie_id')['item_id'].nunique()
n_events = events_filtered.groupby('cookie_id').size()
features = pd.concat([n_events, n_unique_items], axis=1)
features.columns = ['n_events', 'n_unique_items']

In [92]:
event_counts = events_filtered.groupby(['cookie_id', 'event_name']).size()
event_counts = event_counts.unstack(fill_value=0)
event_counts = event_counts.add_prefix('n_')

In [93]:
features = pd.concat([features, event_counts], axis=1)

In [94]:
n_unique_location = events_filtered.groupby('cookie_id')['item_location'].nunique()
n_unique_category = events_filtered.groupby('cookie_id')['item_category'].nunique()

In [95]:
features['n_unique_locations'] = n_unique_location
features['n_unique_categories'] = n_unique_category

In [96]:
features['favourite_share'] = features['n_favorite_add'] / features['n_events']
features

,n_events,n_unique_items,n_contact_chat_open,n_contact_message_sent,n_contact_phone_show,n_favorite_add,n_item_view,n_login,n_photo_swipe,n_search_results_view,n_seller_page_view,n_unique_locations,n_unique_categories,favourite_share
cookie_id,,,,,,,,,,,,,,
ck_00013fdfa0bd37dd,15,9,2,0,0,1,4,2,3,3,0,4,3,0.066667
ck_000c95f1408dcb00,16,9,0,0,0,0,11,0,1,3,1,8,2,0.000000
ck_000e8c52636e3bec,34,20,0,0,1,4,11,1,5,9,3,12,4,0.117647
ck_0010e31baa4a1fb7,16,8,0,0,0,2,7,1,1,4,1,4,3,0.125000
ck_0010ec3874fb5378,74,31,0,0,0,10,25,1,14,20,4,19,2,0.135135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ck_ffee36a37e2532f5,81,38,1,2,6,1,27,0,15,24,5,23,4,0.012346
ck_fff88a7920fd0fb6,11,6,0,0,0,0,8,0,1,2,0,3,1,0.000000
ck_fff9fdc2adf5100b,37,22,0,1,7,0,19,2,4,2,2,11,1,0.000000


In [97]:
new = pd.concat([train, test], ignore_index=True).set_index('cookie_id')
new

,cookie_created_at,window_start_ts,window_end_ts,target
cookie_id,,,,
ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0.0
ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0.0
ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0.0
ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0.0
ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0.0
...,...,...,...,...
ck_8b772caed8f1541c,2026-03-06 06:17:32,2026-04-26,2026-04-27,NaN
ck_5403c76fe68a08ba,2025-09-16 01:17:03,2026-04-26,2026-04-27,NaN
ck_3e6833574edf1736,2026-02-06 19:45:02,2026-04-26,2026-04-27,NaN


In [98]:
features['cookie_age_id'] = (new['window_end_ts'] - new['cookie_created_at']).dt.total_seconds() / 86400
features

,n_events,n_unique_items,n_contact_chat_open,n_contact_message_sent,n_contact_phone_show,n_favorite_add,n_item_view,n_login,n_photo_swipe,n_search_results_view,n_seller_page_view,n_unique_locations,n_unique_categories,favourite_share,cookie_age_id
cookie_id,,,,,,,,,,,,,,,
ck_00013fdfa0bd37dd,15,9,2,0,0,1,4,2,3,3,0,4,3,0.066667,197.215428
ck_000c95f1408dcb00,16,9,0,0,0,0,11,0,1,3,1,8,2,0.000000,233.960440
ck_000e8c52636e3bec,34,20,0,0,1,4,11,1,5,9,3,12,4,0.117647,26.480150
ck_0010e31baa4a1fb7,16,8,0,0,0,2,7,1,1,4,1,4,3,0.125000,83.715069
ck_0010ec3874fb5378,74,31,0,0,0,10,25,1,14,20,4,19,2,0.135135,103.395208
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ck_ffee36a37e2532f5,81,38,1,2,6,1,27,0,15,24,5,23,4,0.012346,112.370810
ck_fff88a7920fd0fb6,11,6,0,0,0,0,8,0,1,2,0,3,1,0.000000,59.496227
ck_fff9fdc2adf5100b,37,22,0,1,7,0,19,2,4,2,2,11,1,0.000000,1.550058


### Временные признаки

In [99]:
events_sorted = events_filtered.sort_values(['cookie_id', 'event_ts'])

In [100]:
events_sorted['seconds_since_prev_event'] = (events_sorted.groupby('cookie_id')['event_ts'].diff().dt.total_seconds())

In [101]:
events_sorted[['cookie_id','event_ts','seconds_since_prev_event']].head(10)

,cookie_id,event_ts,seconds_since_prev_event
67286,ck_00013fdfa0bd37dd,2026-04-24 18:41:16,NaN
72380,ck_00013fdfa0bd37dd,2026-04-24 19:16:53,2137.0
189910,ck_00013fdfa0bd37dd,2026-04-24 19:16:54,1.0
104718,ck_00013fdfa0bd37dd,2026-04-24 19:17:05,11.0
237561,ck_00013fdfa0bd37dd,2026-04-24 20:05:05,2880.0
144174,ck_00013fdfa0bd37dd,2026-04-24 20:31:33,1588.0
324747,ck_00013fdfa0bd37dd,2026-04-24 20:34:06,153.0
249330,ck_00013fdfa0bd37dd,2026-04-24 20:36:30,144.0
278367,ck_00013fdfa0bd37dd,2026-04-24 20:36:32,2.0
28960,ck_00013fdfa0bd37dd,2026-04-24 20:41:33,301.0


In [102]:
features['gap_median'] =(events_sorted.groupby('cookie_id')['seconds_since_prev_event'].median())

In [103]:
features['gap_median'] = features['gap_median'].fillna(0)

In [104]:
features['gap_std'] = (events_sorted.groupby('cookie_id')['seconds_since_prev_event'].std())

### Относительные характеристики поведения

Я использую доли отдельных типов действий и отношения между связанными действиями, чтобы различать куки с одинаковым числом событий, но разной структурой поведения.

In [105]:
event_names = [
    'item_view',
    'search_results_view',
    'photo_swipe',
    'contact_phone_show',
    'contact_chat_open',
    'seller_page_view',
    'contact_message_sent',
    'login',
]

for event_name in event_names:
    features[f'{event_name}_share'] = (features[f'n_{event_name}']/features['n_events'])

In [106]:
features['n_contact_total'] = (features['n_contact_chat_open']+ features['n_contact_message_sent']+features['n_contact_phone_show'])
features['contact_share']= (features['n_contact_total'] / features['n_events'])
features['contact_per_item_view'] = (features['n_contact_total'] /(features['n_item_view'] + 1))
features['photo_per_item_view'] = (features['n_photo_swipe'] / (features['n_item_view'] + 1))
features['search_per_item_view'] = (features['n_search_results_view'] / (features['n_item_view'] + 1))
features['events_per_unique_item'] = (features['n_events'] / (features['n_unique_items'] + 1))

### Признаки движения указателя

Используем разброс и диапазон координат указателя внутри временного окна.
Дополнительно считаем совместный разброс по двум осям и площадь области,
в которой зарегистрированы координаты.

In [107]:
features['pointer_x_std'] = (events_filtered.groupby('cookie_id')['pointer_x'].std())
features['pointer_y_std'] = (events_filtered.groupby('cookie_id')['pointer_y'].std())

In [108]:
features['pointer_x_max'] = events_filtered.groupby('cookie_id')['pointer_x'].max()
features['pointer_x_min'] = events_filtered.groupby('cookie_id')['pointer_x'].min()
features['pointer_y_max'] = events_filtered.groupby('cookie_id')['pointer_y'].max()
features['pointer_y_min'] = events_filtered.groupby('cookie_id')['pointer_y'].min()
features['pointer_x_range'] = features['pointer_x_max'] - features['pointer_x_min']
features['pointer_y_range'] = features['pointer_y_max'] - features['pointer_y_min']
features['pointer_std_norm'] = np.sqrt(features['pointer_x_std']**2 + features['pointer_y_std'] ** 2)
features['pointer_bbox_area'] = features['pointer_x_range'] * features['pointer_y_range']

### Признаки поисковой выдачи

Для событий просмотра поисковой выдачи считаем максимальный, средний и медианный номер страницы, разброс номера страницы и число уникальных просмотренных страниц.

In [109]:
search_events = events_filtered[events_filtered['event_name'] == 'search_results_view']
features['max_search_page'] =search_events.groupby('cookie_id')['search_page'].max()


In [110]:
features['search_page_mean'] = search_events.groupby('cookie_id')['search_page'].mean()
features['search_page_median'] = search_events.groupby('cookie_id')['search_page'].median()
features['search_page_std'] = search_events.groupby('cookie_id')['search_page'].std()
features['n_unique_search_pages'] = search_events.groupby('cookie_id')['search_page'].nunique()

In [111]:
feature_cols = [
    'n_events',
    'n_unique_items',
    'n_contact_chat_open',
    'n_contact_message_sent',
    'n_contact_phone_show',
    'n_item_view',
    'n_login',
    'n_photo_swipe',
    'n_search_results_view',
    'n_unique_locations',
    'n_unique_categories',
    'favourite_share',
    'cookie_age_id',
    'gap_median',
    'gap_std',
    'max_search_page',
    'pointer_x_std',
    'pointer_y_std',
    'item_view_share',
    'search_results_view_share',
    'photo_swipe_share',
    'contact_phone_show_share',
    'contact_chat_open_share',
    'seller_page_view_share',
    'contact_message_sent_share',
    'login_share',
    'n_contact_total',
    'contact_share',
    'contact_per_item_view',
    'photo_per_item_view',
    'search_per_item_view',
    'events_per_unique_item',
    'pointer_x_max',
    'pointer_x_range',
    'pointer_y_range',
    'pointer_std_norm',
    'pointer_bbox_area',
    'search_page_mean',
    'search_page_median',
    'search_page_std',
    'n_unique_search_pages',
]

## Validation

Тестовая выборка расположена во времени после обучающей, поэтому случайное разбиение не соответствует реальному сценарию

Используются четыре последовательных временных фолда:
- 9–10 апреля
- 11–13 апреля
- 14–16 апреля
- 17–19 апреля

Это проверяет устойчивость модели к изменению данных во времени

In [112]:
train.groupby('window_start_ts')['target'].agg(n='size',n_bots='sum',bot_share='mean')

,n,n_bots,bot_share
window_start_ts,,,
2026-04-06,772,68,0.088083
2026-04-07,797,59,0.074028
2026-04-08,834,70,0.083933
2026-04-09,902,66,0.073171
2026-04-10,912,82,0.089912
2026-04-11,896,79,0.088170
2026-04-12,817,67,0.082007
2026-04-13,843,53,0.062871
2026-04-14,826,70,0.084746


In [113]:
def temporal_cv(feature_cols, model):
    cv_folds = [
        (pd.Timestamp('2026-04-09'), pd.Timestamp('2026-04-11')),
        (pd.Timestamp('2026-04-11'), pd.Timestamp('2026-04-14')),
        (pd.Timestamp('2026-04-14'), pd.Timestamp('2026-04-17')),
        (pd.Timestamp('2026-04-17'), pd.Timestamp('2026-04-20')),
    ]

    scores = []

    for valid_start, valid_end in cv_folds:
        train_mask = train['window_start_ts'] < valid_start

        valid_mask = (train['window_start_ts'] >= valid_start)& (train['window_start_ts'] < valid_end)

        train_ids = train.loc[train_mask, 'cookie_id']
        valid_ids = train.loc[valid_mask, 'cookie_id']
        fold_model = CatBoostClassifier(**model.get_params())
        fold_model.fit(features.loc[train_ids, feature_cols],train.loc[train_mask, 'target'])

        pred = fold_model.predict_proba(features.loc[valid_ids, feature_cols])[:, 1]
        scores.append(precision_at_recall(train.loc[valid_mask, 'target'],pred))

    return scores, np.mean(scores)

## Эксперименты и выбор модели

В качестве baseline были рассмотрены несколько классических моделей(логистическая регрессия, градиентный бустинг).

Лучший результат среди начальных вариантов у CatBoost.

Потом за счёт признаков поведения внутри заданного временного окна улучшалось качество. 

Самые полезные:
- временные интервалы между событиями
- абсолютные количества и доли типов событий
- разнообразие объявлений, категорий и локаций
- признаки поисковой выдачи
- характеристики координат указателя

Некоторые признаки я исключила после проверки на временной кросс-валидации (`same_event_repeat_share`, `event_type_entropy` и `n_seller_page_view` ухудшали качество)

Для итоговых признаков были проверены несколько значений глубины дерева и числа итераций CatBoost. Лучший результат(и устойчивый)
получен при:

- `depth = 5`
- `iterations = 800`
- `learning_rate = 0.03`

На четырёх временных фолдах при `random_seed = 42` получились:

`[0.7647, 0.7107, 0.7026, 0.8071]`

Среднее значение Precision@Recall>=0.70 было 0.7463

## Выбранная модель

In [114]:
catboost_model = CatBoostClassifier(
    iterations=800,
    depth=5,
    learning_rate=0.03,
    loss_function='Logloss',
    random_seed=RANDOM_STATE,
    verbose=False
)

In [115]:
scores, mean_score = temporal_cv(feature_cols, catboost_model)
print(scores,mean_score)

[0.7647058823529411, 0.7106598984771574, 0.7025641025641025, 0.8071428571428572] 0.7462681851342645


## Обучение итоговой модели

In [116]:
train_ids = train['cookie_id']
test_ids = test['cookie_id']

X_train = features.loc[train_ids, feature_cols]
y_train = train['target']

X_test = features.loc[test_ids, feature_cols]

In [117]:
final_model = CatBoostClassifier(
    iterations=800,
    depth=5,
    learning_rate=0.03,
    loss_function='Logloss',
    random_seed=RANDOM_STATE,
    verbose=False
)

final_model.fit(X_train, y_train)

test_pred = final_model.predict_proba(X_test)[:, 1]

In [118]:
print(X_train.shape, X_test.shape)
print(len(test_pred))
print(test_pred.min(),test_pred.max(),test_pred.mean())
print(np.isnan(test_pred).sum())

(11091, 41) (4909, 41)
4909
0.00016941597571922036 0.9991176697760368 0.08453321091514884
0


In [119]:
submission = pd.DataFrame({
    'cookie_id': test['cookie_id'],
    'score': test_pred,
})

assert len(submission) == len(test)
assert submission['cookie_id'].equals(test['cookie_id'])
assert submission['cookie_id'].is_unique
assert submission['score'].between(0, 1).all()
assert submission['score'].notna().all()

submission.to_csv('submission.csv', index=False)